 # Workflow for a transformation pathway of a single node energy system with perfect foresight including material flows

 In this application of the ETHOS.FINE framework, a transformation pathway of a energy system is modeled and optimized.

 All classes which are available to the user are utilized and examples of the selection of different parameters within these classes are given.

 The workflow is structures as follows:
 1. Required packages are imported and the input data path is set
 2. An energy system model instance is created
 3. Commodity sources are added to the energy system model
 4. Commodity conversion components are added to the energy system model
 5. Commodity storages are added to the energy system model
 7. Commodity sinks are added to the energy system model
 8. Material sinks are added to the energy system model
 9. Material sources are addeed to the energy system model
 10. Material conversions are added to the energy system model for recycling processes
 11. The energy system model is optimized
 12. Selected optimization results are presented


 # 1. Import required packages and set input data path

 The ETHOS.FINE framework is imported which provides the required classes and functions for modeling the energy system.

In [24]:
import fine as fn
from getData import getData
import pandas as pd
import os

cwd = os.getcwd()
data = getData()

 # 2. Create an energy system model instance

 The structure of the energy system model is given by the considered locations, commodities, the number of time steps as well as the hours per time step.

 The commodities are specified by a unit (i.e. 'GW_electric', 'GW_H2lowerHeatingValue', 'Mio. t CO2/h') which can be given as an energy or mass unit per hour. Furthermore, the cost unit and length unit are specified.

In [25]:
locations = {"GermanyRegion"}
commodityUnitsDict = {
    "electricity": r"GW$_{el}$",
    "hydrogen": r"GW$_{H_{2},LHV}$",
}
commodities = {"electricity", "hydrogen"}
materials = {
    "lithium", "cobalt", "nickel", "graphite",
    "nmc_lithium_scrap", "nmc_graphite_scrap", "nmc_cobalt_scrap", "nmc_nickel_scrap",
    "lfp_lithium_scrap", "lfp_graphite_scrap",
}

materialUnitsDict = {
    "lithium": "tons",
    "graphite": "tons",
    "cobalt": "tons",
    "nickel": "tons",
    "nmc_lithium_scrap": "Megatons",
    "nmc_graphite_scrap": "Megatons",
    "nmc_cobalt_scrap": "Megatons",
    "nmc_nickel_scrap": "Megatons",
    "lfp_lithium_scrap": "Megatons",
    "lfp_graphite_scrap": "Megatons",
}



numberOfTimeSteps = 8760
hoursPerTimeStep = 1



 # 2.1 define Transformation Pathway parameters

 Transformation Pathway Analyses can be run by setting a number of investment periods
 larger than 1, which is the default value and results in a single year optimization.

In [26]:
numberOfInvestmentPeriods=3
startYear=2025
interval=5

In [27]:
pathwayBalanceLimit = pd.DataFrame(columns=["GermanyRegion", "Total", "lowerBound"], index=["Cobalt_Resources", "Lithium_Resources", "Graphite_Resources", "Nickel_Resources"])
pathwayBalanceLimit.loc["Cobalt_Resources"] = [None, 550, False]
pathwayBalanceLimit.loc["Lithium_Resources"] = [None, 530, False]
pathwayBalanceLimit.loc["Graphite_Resources"] = [None, 3400, False]
pathwayBalanceLimit.loc["Nickel_Resources"] = [None, 3400, False]

In [28]:
esM = fn.EnergySystemModel(
    locations=locations,
    commodities=commodities,
    materials=materials,
    numberOfInvestmentPeriods=numberOfInvestmentPeriods,
    startYear=startYear, 
    investmentPeriodInterval=interval,
    numberOfTimeSteps=8760,
    commodityUnitsDict=commodityUnitsDict,
    materialUnitsDict=materialUnitsDict,
    hoursPerTimeStep=1,
    costUnit="1e9 Euro",
    lengthUnit="km",
    verboseLogLevel=0,
    pathwayBalanceLimit=pathwayBalanceLimit,
)

DEBUG: type(self.materialUnitsDict) = <class 'dict'>
DEBUG: self.materialUnitsDict = {'lithium': 'tons', 'graphite': 'tons', 'cobalt': 'tons', 'nickel': 'tons', 'nmc_lithium_scrap': 'Megatons', 'nmc_graphite_scrap': 'Megatons', 'nmc_cobalt_scrap': 'Megatons', 'nmc_nickel_scrap': 'Megatons', 'lfp_lithium_scrap': 'Megatons', 'lfp_graphite_scrap': 'Megatons'}
DEBUG: type(self.materials) = <class 'set'>
DEBUG: self.materials = {'lithium', 'cobalt', 'graphite', 'nmc_cobalt_scrap', 'nmc_lithium_scrap', 'nmc_nickel_scrap', 'nickel', 'lfp_graphite_scrap', 'nmc_graphite_scrap', 'lfp_lithium_scrap'}


 # 3. Add commodity sources to the energy system model

 ## 3.1. Electricity sources

 ### Wind

 change weather conditions for the different investment periods

In [29]:
operationRateMax={}
operationRateMax[2025]=1.2*data["Wind (onshore), operationRateMax"]
operationRateMax[2030]=0.7*data["Wind (onshore), operationRateMax"]
operationRateMax[2035]=1*data["Wind (onshore), operationRateMax"]


 add wind onshore source to esM

In [30]:
import pandas as pd

esM.add(
    fn.Source(
        esM=esM,
        name="Wind (onshore)",
        commodity="electricity",
        hasCapacityVariable=True,
        operationRateMax=data["Wind (onshore), operationRateMax"],
        capacityMax=data["Wind (onshore), capacityMax"],
        investPerCapacity={2025:1.18, 2030:1.15, 2035:1.13},
        opexPerCapacity=0.024,
        interestRate=0.08,
        economicLifetime=25,
    )
)


In [31]:
operationRateMax={}
operationRateMax[2025]=1.2*data["Wind (offshore), operationRateMax"]
operationRateMax[2030]=0.7*data["Wind (offshore), operationRateMax"]
operationRateMax[2035]=1*data["Wind (offshore), operationRateMax"]


In [32]:
esM.add(
    fn.Source(
        esM=esM,
        name="Wind (offshore)",
        commodity="electricity",
        hasCapacityVariable=True,
        operationRateMax=data["Wind (offshore), operationRateMax"],
        capacityMax=data["Wind (offshore), capacityMax"],
        investPerCapacity={2025:1.18, 2030:1.15, 2035:1.13},
        opexPerCapacity=0.024,
        interestRate=0.08,
        economicLifetime=25,
    )
)

 # 4. Add conversion components to the energy system model

 ### Electrolyzers

 add component with constant invest and opex per capacity

In [33]:
esM.add(
    fn.Conversion(
        esM=esM,
        name="Electroylzers",
        physicalUnit=r"GW$_{el}$",
        commodityConversionFactors={"electricity": -1, "hydrogen": 0.7},
        hasCapacityVariable=True,
        investPerCapacity=0.5,
        opexPerCapacity=0.5 * 0.025,
        interestRate=0.08,
        economicLifetime=10,
    )
)

 # 5. Add commodity storages to the energy system model

 ### Lithium ion batteries

 The self discharge of a lithium ion battery is here described as 3% per month. The self discharge per hours is obtained using the equation (1-$\text{selfDischarge}_\text{hour})^{30*24\text{h}} = 1-\text{selfDischarge}_\text{month}$.

In [34]:
esM.add(
    fn.Storage(
        esM=esM,
        name="nmc",
        commodity="electricity",
        hasCapacityVariable=True,
        chargeEfficiency=0.96,
        cyclicLifetime=3000,
        dischargeEfficiency=0.96,
        selfDischarge=1 - (1 - 0.03) ** (1 / (30 * 24)),
        chargeRate=1,
        dischargeRate=1,
        doPreciseTsaModeling=False,
        investPerCapacity={2025:0.4275,2030: 0.3774, 2035:0.3468},
        opexPerCapacity=0.025,
        interestRate=0.08,
        economicLifetime=10,
        materialIntensity = {
            'GermanyRegion': {
                'lithium': pd.Series({ 2015: 0.150, 2020: 0.149, 2025: 0.146, 2030: 0.145, 2035: 0.145}, dtype='float64'),
                'nickel': pd.Series({ 2015: 0.530, 2020: 0.567, 2025: 0.622, 2030: 0.640, 2035: 0.648}, dtype='float64'),
                'cobalt': pd.Series({ 2015: 0.177, 2020: 0.153, 2025: 0.117, 2030: 0.105, 2035: 0.099}, dtype='float64'),
                'graphite': pd.Series({ 2015: 1.064, 2020: 0.971, 2025: 0.831, 2030: 0.784, 2035: 0.763}, dtype='float64'),
 
            },
        },
        materialRecovery = {
            'GermanyRegion': {
                'lithium': pd.Series({2025: 0.55, 2030: 0.73, 2035: 0.84}, dtype='float64'),
                'nickel': pd.Series({2025: 0.55, 2030: 0.73, 2035: 0.84}, dtype='float64'),
                'cobalt': pd.Series({2025: 0.55, 2030: 0.73, 2035: 0.84}, dtype='float64'),
                'graphite': pd.Series({2025: 0.55, 2030: 0.73, 2035: 0.84}, dtype='float64'),
            },
        }
    )
)

In [35]:
esM.add(
    fn.Storage(
        esM=esM,
        name="lfp",
        commodity="electricity",
        hasCapacityVariable=True,
        chargeEfficiency=0.96,
        cyclicLifetime=5000,
        dischargeEfficiency=0.96,
        selfDischarge=1 - (1 - 0.03) ** (1 / (30 * 24)),
        chargeRate=1,
        dischargeRate=1,
        doPreciseTsaModeling=False,
        investPerCapacity={2025:0.3485,2030: 0.2746, 2035:0.2552},
        opexPerCapacity=0.025,
        interestRate=0.08,
        economicLifetime=17,
        materialIntensity = {
            'GermanyRegion': {
                'lithium': pd.Series({2010: 0.16, 2015: 0.16, 2020: 0.16, 2025: 0.16, 2030: 0.16, 2035: 0.16}, dtype='float64'),
                'graphite': pd.Series({2010: 1.19, 2015: 1.19, 2020: 1.19, 2025: 1.19, 2030: 1.19, 2035: 1.19}, dtype='float64'),
 
            },
        },
        materialRecovery = {
            'GermanyRegion': {
                'lithium': pd.Series({2025: 0.55, 2030: 0.73, 2035: 0.84}, dtype='float64'),
                'graphite': pd.Series({2025: 0.55, 2030: 0.73, 2035: 0.84}, dtype='float64'),
            },
        }
    )
)

 ## 5.2. Hydrogen storage

 ### Hydrogen filled salt caverns
 The maximum capacity is here obtained by: dividing the given capacity (which is given for methane) by the lower heating value of methane and then multiplying it with the lower heating value of hydrogen.

In [36]:
esM.add(
    fn.Storage(
        esM=esM,
        name="Salt caverns (hydrogen)",
        commodity="hydrogen",
        hasCapacityVariable=True,
        capacityVariableDomain="continuous",
        capacityPerPlantUnit=133,
        chargeRate=1 / 470.37,
        dischargeRate=1 / 470.37,
        sharedPotentialID="Existing salt caverns",
        stateOfChargeMin=0.33,
        stateOfChargeMax=1,
        capacityMax=data["Salt caverns (hydrogen), capacityMax"],
        investPerCapacity={2025:0.00011,2030: 0.00009,2035:0.00009},
        opexPerCapacity=0.00057,
        interestRate=0.08,
        economicLifetime=30,
    )
)

 # 6. Add commodity sinks to the energy system model

 ## 6.1. Electricity sinks

 ### Electricity demand

 vary the demand with the years - increasing demand by 30% per year

In [37]:
electricityDemand={}
electricityDemand[2025]=(1+0*0.3)*data["Electricity demand, operationRateFix"]
electricityDemand[2030]=(1+1*0.3)*data["Electricity demand, operationRateFix"]
electricityDemand[2035]=(1+2*0.3)*data["Electricity demand, operationRateFix"]

esM.add(
    fn.Sink(
        esM=esM,
        name="Electricity demand",
        commodity="electricity",
        hasCapacityVariable=False,
        operationRateFix=electricityDemand,
    )
)

 ## 6.2. Hydrogen sinks

 ### Fuel cell electric vehicle (FCEV) demand

In [38]:
FCEV_penetration = 0.5

# vary the demand with the years - increasing demand by 25% per year
hydrogendDemand={}
hydrogendDemand[2025]=(1+0*0.25)*data["Hydrogen demand, operationRateFix"] * FCEV_penetration
hydrogendDemand[2030]=(1+0*0.25)*data["Hydrogen demand, operationRateFix"] * FCEV_penetration
hydrogendDemand[2035]=(1+0*0.25)*data["Hydrogen demand, operationRateFix"] * FCEV_penetration


esM.add(
    fn.Sink(
        esM=esM,
        name="Hydrogen demand",
        commodity="hydrogen",
        hasCapacityVariable=False,
        operationRateFix=hydrogendDemand,
    )
)

 # 7. Add material sinks to the energy system model

In [39]:
sink = esM.add(
    fn.Sink(
        esM=esM,
        name="Lithium demand",
        hasCapacityVariable=False,
        commodity="lithium",
        material=True,      
    )
)

sink = esM.add(
    fn.Sink(
        esM=esM,
        name="Cobalt demand",
        hasCapacityVariable=False,
        commodity="cobalt",
        material=True,      
    )
)

sink = esM.add(
    fn.Sink(
        esM=esM,
        name="Nickel demand",
        hasCapacityVariable=False,
        commodity="nickel",
        material=True,      
    )
)

sink = esM.add(
    fn.Sink(
        esM=esM,
        name="Graphite demand",
        hasCapacityVariable=False,
        commodity="graphite",
        material=True,      
    )
)


 # 8. Add material sources to the energy system model

## 8.1 Add primary material sources with pathway balance limit

In [40]:
esM.add(
    fn.Source(
        esM=esM,
        name="Lithium supply",
        hasCapacityVariable=False,
        commodity="lithium",
        pathwayBalanceLimitID="Lithium_Resources"
    )
)

esM.add(
    fn.Source(
        esM=esM,
        name="Cobalt supply",
        hasCapacityVariable=False,
        commodity="cobalt",
        pathwayBalanceLimitID="Cobalt_Resources"
    )
)

esM.add(
    fn.Source(
        esM=esM,
        name="Graphite supply",
        hasCapacityVariable=False,
        commodity="graphite",
        pathwayBalanceLimitID="Graphite_Resources"
    )
)

esM.add(
    fn.Source(
        esM=esM,
        name="Nickel supply",
        hasCapacityVariable=False,
        commodity="nickel",
        pathwayBalanceLimitID="Nickel_Resources"
    )
)

## 8.2 Add secondary material sources per component and material

In [41]:
esM.add(
    fn.Source(
        esM=esM,
        name="nmc_lithium_scrap",
        commodity="nmc_lithium_scrap",
        hasCapacityVariable=False,
        material=True
    )
)

esM.add(
    fn.Source(
        esM=esM,
        name="nmc_cobalt_scrap",
        commodity="nmc_cobalt_scrap",
        hasCapacityVariable=False,
        material=True
    )
)

esM.add(
    fn.Source(
        esM=esM,
        name="nmc_nickel_scrap",
        commodity="nmc_nickel_scrap",
        hasCapacityVariable=False,
        material=True
    )
)

esM.add(
    fn.Source(
        esM=esM,
        name="nmc_graphite_scrap",
        commodity="nmc_graphite_scrap",
        hasCapacityVariable=False,
        material=True
    )
)

esM.add(
    fn.Source(
        esM=esM,
        name="lfp_lithium_scrap",
        commodity="lfp_lithium_scrap",
        hasCapacityVariable=False,
        material=True
    )
)

esM.add(
    fn.Source(
        esM=esM,
        name="lfp_graphite_scrap",
        commodity="lfp_graphite_scrap",
        hasCapacityVariable=False,
        material=True
    )
)

# 9. Add Recycling plants 

In [42]:
esM.add(
    fn.Conversion(
        esM=esM,
        name="NMC Onshore",
        physicalUnit=r"GW$_{el}$",
        commodityConversionFactors={"nmc_lithium_scrap": -1, "lithium":0.9, 
                                    "nmc_cobalt_scrap": -1, "cobalt":0.98, 
                                    "nmc_graphite_scrap": -1, "graphite":0.9, 
                                    "nmc_nickel_scrap": -1, "nickel":0.98, 
                                    },
        hasCapacityVariable=True,
        investPerCapacity=0.7,
        opexPerCapacity=0.021,
        interestRate=0.08,
        economicLifetime=33,
    )
)

esM.add(
    fn.Conversion(
        esM=esM,
        name="LFP Onshore",
        physicalUnit=r"GW$_{el}$",
        commodityConversionFactors={"lfp_lithium_scrap": -1, "lithium":0.9, 
                                    "lfp_graphite_scrap": -1, "graphite":0.9, 
                                    },
        hasCapacityVariable=True,
        investPerCapacity=0.7,
        opexPerCapacity=0.021,
        interestRate=0.08,
        economicLifetime=33,
    )
)

 # 10. Optimize energy system model

 All components are now added to the model and the model can be optimized. If the computational complexity of the optimization should be reduced, the time series data of the specified components can be clustered before the optimization and the parameter timeSeriesAggregation is set to True in the optimize call.

In [43]:
#esM.generationSecondaryMaterialSources()
#esM.generationMaterialSinks()

In [44]:
esM.aggregateTemporally(numberOfTypicalPeriods=30)


Clustering time series data with 30 typical periods and 24 time steps per period 
further clustered to 12 segments per period...
		(14.4532 sec)



In [45]:
esM.optimize(timeSeriesAggregation=True, solver="gurobi")

Time series aggregation specifications:
Number of typical periods:30, number of time steps per period:24, number of segments per period:12

Declaring sets, variables and constraints for SourceSinkModel
	declaring sets... 
	declaring variables... 
	declaring constraints... 
		(1.0884 sec)

Declaring sets, variables and constraints for ConversionModel
	declaring sets... 
	declaring variables... 
	declaring constraints... 
		(0.2430 sec)

Declaring sets, variables and constraints for StorageModel
	declaring sets... 
	declaring variables... 
	declaring constraints... 
		(1.9261 sec)

Declaring shared potential constraint...
		(0.0003 sec)

Declaring linked component quantity constraint...
		(0.0000 sec)

Declaring commodity balances...
		(0.5708 sec)

Declaring material balances...
('GermanyRegion', 'Lithium demand', 'lithium', 0)
('GermanyRegion', 'Lithium demand', 'lithium', 1)
('GermanyRegion', 'Lithium demand', 'lithium', 2)
('GermanyRegion', 'Cobalt demand', 'cobalt', 0)
('GermanyRegi

 # 11. Selected results output

 ### Sources and Sink

 Show optimization summary

In [46]:
for year in [2025,2030,2035]:
    print(f"\n Results of SourceSinkModel for year {year}")
    print(esM.getOptimizationSummary("SourceSinkModel", outputLevel=2, ip=year))


 Results of SourceSinkModel for year 2025


KeyError: 2025

 ### Conversion

 Show optimization summary

In [ ]:
for year in [2025,2030,2035]:
    print(f"\n Results of ConversionMpdel for year {year}")
    print(esM.getOptimizationSummary("ConversionModel", outputLevel=2, ip=year))


 Results of ConversionMpdel for year 2025
                                              GermanyRegion
Component     Property        Unit                         
Electroylzers NPVcontribution [1e9 Euro]            0.49069
              TAC             [1e9 Euro/a]         0.113793
              capacity        [GW$_{el}$]          1.307744
              capexCap        [1e9 Euro/a]         0.097446
              commissioning   [GW$_{el}$]          1.307744
              invest          [1e9 Euro]           0.653872
              operation       [GW$_{el}$*h/a]   6807.249824
                              [GW$_{el}$*h]     6807.249824
              opexCap         [1e9 Euro/a]         0.016347

 Results of ConversionMpdel for year 2030
                                              GermanyRegion
Component     Property        Unit                         
Electroylzers NPVcontribution [1e9 Euro]           0.416315
              TAC             [1e9 Euro/a]         0.141857
              

 Operation color map for New CCGT plants (hydrogen) in Investment Period 2020

 Operation color map for New CCGT plants (hydrogen) in Investment Period 2030

 ### Storage

 Show optimization summary

In [ ]:
for year in [2025,2030,2035]:
    print(f"\n Results of StorageModel for year {year}")
    print(esM.getOptimizationSummary("StorageModel", outputLevel=2, ip=year))


 Results of StorageModel for year 2025
                                                                               GermanyRegion
Component               Property                        Unit                                
Salt caverns (hydrogen) NPVcontribution                 [1e9 Euro]                  1.394318
                        TAC                             [1e9 Euro/a]                0.323348
                        capacity                        [GW$_{H_{2},LHV}$*h]       557.71676
                        capexCap                        [1e9 Euro/a]                0.005449
                        commissioning                   [GW$_{H_{2},LHV}$*h]       557.71676
                        invest                          [1e9 Euro]                  0.061349
                        operationCharge                 [GW$_{H_{2},LHV}$*h/a]    2092.43811
                                                        [GW$_{H_{2},LHV}$*h]      2092.43811
                        operat